# ImageDETECTIVE: fakes from open models (Kaggle)

Generates AI images with **PixArt-Σ** (GPU 0) and **SD 3.5 Medium** (GPU 1) from the pair captions in the repo,
so the detector sees more generator architectures. All logic is in `scripts/generate_open_models.py`.

**Settings before running** (right panel):
1. *Accelerator*: **GPU T4 x2**
2. *Internet*: **on** (clone the repo, install diffusers, download the models)
3. *Add-ons → Secrets*: a secret named **`HF_TOKEN`** with your Hugging Face token, attached to this notebook
4. On Hugging Face, accept the terms of `stabilityai/stable-diffusion-3.5-medium` once (same account as the token)

**Two steps:** first run all cells with `LIMIT = 3` (quick test, a few minutes). If the images look right,
set `LIMIT = None` (or a number, if the test shows the full run would take more than ~11 hours), then
**Save Version → Save & Run All (Commit)**. The images end up in the version's output: `open_gen/`.

In [ ]:
LIMIT = 3  # quick test: 3 images per model. Full run: None (or e.g. 300 per model)

In [ ]:
import os

os.environ["HF_HOME"] = "/tmp/hf"  # model downloads stay out of the notebook output
!rm -rf /tmp/repo && git clone --depth 1 https://github.com/phantom007xy/ImageDETECTIVE.git /tmp/repo
# Kaggle's preinstalled torchao is too old for diffusers 0.41 (no FqnToConfig) and crashes its import;
# we use no quantization, so without torchao diffusers simply skips that part.
!pip uninstall -y -q torchao
!pip install -q diffusers==0.41.0 sentencepiece

from kaggle_secrets import UserSecretsClient

os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")  # never printed
!nvidia-smi --query-gpu=index,name,memory.total --format=csv

In [ ]:
import subprocess
import sys
import time

OUT = "/kaggle/working/open_gen"
os.makedirs(OUT, exist_ok=True)
common = [sys.executable, "scripts/generate_open_models.py", "--prompts", "data/prompts/pairs_v1_prompts.csv",
          "data/prompts/pairs_v2_prompts.csv", "--parts", "2", "--out-dir", OUT]
if LIMIT is not None:
    common += ["--limit", str(LIMIT)]


def start(model, part, device):
    log = open(f"{OUT}/{model}.log", "w")
    return subprocess.Popen(common + ["--model", model, "--part", str(part), "--device", device],
                            cwd="/tmp/repo", stdout=log, stderr=subprocess.STDOUT)


# PixArt first; SD 3.5 only once PixArt is generating, so both never load into RAM at the same time.
pixart = start("pixart", 0, "cuda:0")
while pixart.poll() is None and "[1/" not in open(f"{OUT}/pixart.log").read():
    time.sleep(10)
sd35m = start("sd35m", 1, "cuda:1")
print("exit codes: pixart", pixart.wait(), "| sd35m", sd35m.wait())
!tail -n 6 {OUT}/pixart.log {OUT}/sd35m.log

In [ ]:
import csv
import glob
from collections import Counter

from PIL import Image

for model in ("pixart", "sd35m"):
    path = f"{OUT}/{model}/manifest.csv"
    rows = list(csv.DictReader(open(path))) if os.path.exists(path) else []
    seconds = [float(r["seconds"]) for r in rows]
    print(model, dict(Counter(r["status"] for r in rows)),
          f"| {sum(seconds) / max(1, len(seconds)):.1f} s per image" if seconds else "")
    for p in sorted(glob.glob(f"{OUT}/{model}/*.png"))[:3]:  # a few images to look at
        im = Image.open(p)
        display(im.resize((384, round(384 * im.height / im.width))))